# Fine-tuning the Wasting Detection Models with Real Data

This notebook mixes your real labeled measurements with the existing synthetic training
dataset and retrains both models:
- **Weight estimator** (regression — predicts weight from body proportions)
- **Wasting classifier** (5-class — SAM / MAM / Normal / Risk_Overweight / Overweight)

## Prerequisites
1. Run `scripts/batch_assess.py` on your labeled photos to generate `data/batch_results.csv`
2. That CSV must have a non-empty `finetune_label` column (auto-filled when ground-truth height+weight are provided)
3. The synthetic training dataset must exist at `data/training_data/synthetic_dataset.csv`  
   (run `python ml/generate_synthetic_data.py` if missing)

## What happens here
Real data is repeated `REAL_DATA_REPEAT` times before merging with synthetic data.  
This upweights real samples without discarding the synthetic baseline.

In [ ]:
import sys
from pathlib import Path

# Make sure we can import from the project root
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import warnings
warnings.filterwarnings('ignore')

from ml.models import FEATURE_NAMES, WASTING_LABELS, build_weight_estimator, build_wasting_classifier

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR        = PROJECT_ROOT / "data"
MODELS_DIR      = DATA_DIR / "models"
REAL_DATA_CSV   = DATA_DIR / "batch_results.csv"       # output of batch_assess.py
SYNTH_DATA_CSV  = DATA_DIR / "training_data" / "synthetic_dataset.csv"

# ── Hyperparameters ──────────────────────────────────────────────────────────
REAL_DATA_REPEAT = 20    # repeat real samples this many times (upweighting)
                         # increase if real dataset is very small (<20 samples)
EPOCHS           = 150
BATCH_SIZE       = 128
RANDOM_SEED      = 42

print(f"Project root : {PROJECT_ROOT}")
print(f"Real data    : {REAL_DATA_CSV}")
print(f"Synthetic    : {SYNTH_DATA_CSV}")

## 1. Load and inspect real labeled data

In [ ]:
# Load batch_results.csv and keep only rows with a valid label
if not REAL_DATA_CSV.exists():
    raise FileNotFoundError(
        f"{REAL_DATA_CSV} not found.\n"
        "Run: PYTHONPATH=. .venv/bin/python scripts/batch_assess.py --images /path/to/photos/ "
        "--ground-truth data/ground_truth.csv"
    )

real_raw = pd.read_csv(REAL_DATA_CSV)
print(f"Loaded {len(real_raw)} rows from batch_results.csv")

# Keep only rows with a valid finetune_label and all required features
feat_cols = FEATURE_NAMES
real_feat_cols = [
    "feat_shoulder_width_cm", "feat_hip_width_cm", "feat_torso_length_cm",
    "feat_upper_arm_length_cm", "feat_shoulder_height_ratio",
    "feat_hip_height_ratio", "feat_body_build_score",
    "age_months", "sex", "pred_height_cm",   # for feature assembly
    "actual_height_cm", "actual_weight_kg",  # for weight estimator labels
    "finetune_label",
]

real_labeled = real_raw.dropna(subset=["finetune_label", "feat_shoulder_width_cm"]).copy()
real_labeled = real_labeled[real_labeled["finetune_label"].isin(WASTING_LABELS)]

print(f"{len(real_labeled)} rows with valid labels and pose features")
if len(real_labeled) == 0:
    raise ValueError(
        "No labeled rows found. Make sure batch_assess.py was run with --ground-truth "
        "and that actual_height_cm + actual_weight_kg are filled in."
    )

print("\nLabel distribution:")
print(real_labeled["finetune_label"].value_counts())

real_labeled.head()

In [ ]:
# Assemble feature matrix from batch_results columns
def assemble_real_features(df):
    """Map batch_assess output columns → FEATURE_NAMES order."""
    sex_binary = (df["sex"].str.upper() == "M").astype(int)
    height_cm  = df["actual_height_cm"].fillna(df["pred_height_cm"])

    return pd.DataFrame({
        "age_months":            df["age_months"],
        "sex_binary":            sex_binary,
        "height_cm":             height_cm,
        "shoulder_width_cm":     df["feat_shoulder_width_cm"],
        "hip_width_cm":          df["feat_hip_width_cm"],
        "torso_length_cm":       df["feat_torso_length_cm"],
        "upper_arm_length_cm":   df["feat_upper_arm_length_cm"],
        "shoulder_height_ratio": df["feat_shoulder_height_ratio"],
        "hip_height_ratio":      df["feat_hip_height_ratio"],
        "body_build_score":      df["feat_body_build_score"],
    })[FEATURE_NAMES]

X_real  = assemble_real_features(real_labeled).values.astype("float32")
y_real_label  = real_labeled["finetune_label"].values
y_real_weight = real_labeled["actual_weight_kg"].values.astype("float32")  # NaN where unknown

print(f"Real feature matrix: {X_real.shape}")

## 2. Load synthetic training data

In [ ]:
if not SYNTH_DATA_CSV.exists():
    raise FileNotFoundError(
        f"{SYNTH_DATA_CSV} not found. Run:\n"
        "PYTHONPATH=. .venv/bin/python ml/generate_synthetic_data.py"
    )

synth = pd.read_csv(SYNTH_DATA_CSV)
print(f"Synthetic dataset: {len(synth):,} rows")
print("Label distribution:")
print(synth["label"].value_counts())

## 3. Merge real + synthetic data

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Build combined dataset
# Repeat real rows to upweight them vs the large synthetic set
X_real_rep   = np.tile(X_real, (REAL_DATA_REPEAT, 1))
y_label_rep  = np.tile(y_real_label, REAL_DATA_REPEAT)
y_weight_rep = np.tile(y_real_weight, REAL_DATA_REPEAT)

X_synth  = synth[FEATURE_NAMES].values.astype("float32")
y_synth_label  = synth["label"].values
y_synth_weight = synth["weight_kg"].values.astype("float32")

X_all      = np.vstack([X_synth, X_real_rep])
y_all_lbl  = np.concatenate([y_synth_label, y_label_rep])
y_all_wt   = np.concatenate([y_synth_weight, y_weight_rep])

print(f"Combined dataset: {len(X_all):,} rows  "
      f"({len(X_synth):,} synthetic + {len(X_real_rep):,} real×{REAL_DATA_REPEAT})")

# Encode labels
le = LabelEncoder()
le.fit(WASTING_LABELS)
y_all_cls = le.transform(y_all_lbl).astype("int32")

print("\nMerged label distribution:")
for lbl in WASTING_LABELS:
    n = (y_all_lbl == lbl).sum()
    pct = 100 * n / len(y_all_lbl)
    print(f"  {lbl:20s}: {n:6,}  ({pct:.1f}%)")

In [ ]:
# Train / validation split (stratified)
X_tr, X_va, yw_tr, yw_va, yc_tr, yc_va = train_test_split(
    X_all, y_all_wt, y_all_cls,
    test_size=0.15, random_state=RANDOM_SEED, stratify=y_all_cls
)

# Feature scaling
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr).astype("float32")
X_va_s = scaler.transform(X_va).astype("float32")

print(f"Train: {len(X_tr):,}   Val: {len(X_va):,}")

## 4. Retrain the weight estimator

In [ ]:
import tensorflow as tf

# Filter NaN weights (rows where actual_weight was unknown)
mask_tr = ~np.isnan(yw_tr)
mask_va = ~np.isnan(yw_va)

print(f"Weight estimator training rows: {mask_tr.sum():,} / {len(yw_tr):,} "
      f"(real weight missing for {(~mask_tr).sum()} rows)")

we_model = build_weight_estimator()
we_model.fit(
    X_tr_s[mask_tr], yw_tr[mask_tr],
    validation_data=(X_va_s[mask_va], yw_va[mask_va]),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=12, restore_best_weights=True, monitor="val_mae"),
        tf.keras.callbacks.ReduceLROnPlateau(patience=6, factor=0.5, verbose=0),
    ],
    verbose=0,
)

we_val_mae = we_model.evaluate(X_va_s[mask_va], yw_va[mask_va], verbose=0)[1]
print(f"Weight estimator val MAE: {we_val_mae:.3f} kg")

## 5. Retrain the wasting classifier

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Class weights — SAM gets 2× extra penalty (dangerous false negative)
classes = np.unique(yc_tr)
base_weights = compute_class_weight("balanced", classes=classes, y=yc_tr)
cw_dict = dict(zip(classes.tolist(), base_weights.tolist()))
sam_idx = sorted(WASTING_LABELS).index("SAM")
if sam_idx in cw_dict:
    cw_dict[sam_idx] *= 2.0

wc_model = build_wasting_classifier()
history = wc_model.fit(
    X_tr_s, yc_tr,
    validation_data=(X_va_s, yc_va),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=cw_dict,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=12, restore_best_weights=True, monitor="val_accuracy"),
        tf.keras.callbacks.ReduceLROnPlateau(patience=6, factor=0.5, verbose=0),
    ],
    verbose=0,
)

wc_val_acc = max(history.history["val_accuracy"])
print(f"Wasting classifier val accuracy: {wc_val_acc:.3f}")

# Training curve
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history["loss"],     label="train")
ax[0].plot(history.history["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(history.history["accuracy"],     label="train")
ax[1].plot(history.history["val_accuracy"], label="val")
ax[1].set_title("Accuracy"); ax[1].legend()
plt.tight_layout(); plt.show()

## 6. Evaluate — focus on SAM recall

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_pred_probs = wc_model.predict(X_va_s, verbose=0)
y_pred_cls   = np.argmax(y_pred_probs, axis=1)
y_pred_names = le.inverse_transform(y_pred_cls)
y_true_names = le.inverse_transform(yc_va)

print(classification_report(y_true_names, y_pred_names, target_names=sorted(WASTING_LABELS)))

# Confusion matrix
cm = confusion_matrix(y_true_names, y_pred_names, labels=sorted(WASTING_LABELS))
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=sorted(WASTING_LABELS),
            yticklabels=sorted(WASTING_LABELS), cmap="Blues", ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix (fine-tuned model)")
plt.tight_layout(); plt.show()

# SAM recall check
sam_mask   = (y_true_names == "SAM")
if sam_mask.sum() > 0:
    sam_recall = (y_pred_names[sam_mask] == "SAM").mean()
    flag = "✅" if sam_recall >= 0.80 else "⚠️  BELOW TARGET"
    print(f"\nSAM recall: {sam_recall:.2%}  {flag}  (target ≥ 80%)")
else:
    print("\nNo SAM cases in validation set (check your synthetic data distribution).")

## 7. Evaluate on REAL-ONLY held-out samples

The validation set above is mostly synthetic. Let's check performance specifically on your real photos.

In [ ]:
if len(real_labeled) >= 5:
    # Use all real samples for a quick check (small n, so no hold-out split)
    X_real_s = scaler.transform(X_real).astype("float32")
    probs_real = wc_model.predict(X_real_s, verbose=0)
    pred_real  = le.inverse_transform(np.argmax(probs_real, axis=1))
    true_real  = y_real_label

    results_df = pd.DataFrame({
        "image":    real_labeled["image_file"].values,
        "actual":   true_real,
        "predicted": pred_real,
        "correct":  true_real == pred_real,
        "sam_prob": probs_real[:, sorted(WASTING_LABELS).index("SAM")].round(3),
        "mam_prob": probs_real[:, sorted(WASTING_LABELS).index("MAM")].round(3),
    })
    print(f"Real-data accuracy: {results_df['correct'].mean():.0%}  ({results_df['correct'].sum()}/{len(results_df)})\n")
    print(results_df.to_string(index=False))
else:
    print(f"Only {len(real_labeled)} labeled real samples — results are for illustration only.\n"
          "Collect more labeled photos for meaningful evaluation.")

## 8. Save updated models

In [ ]:
# Save updated Keras models + preprocessing artifacts
MODELS_DIR.mkdir(parents=True, exist_ok=True)

we_model.save(MODELS_DIR / "weight_estimator.keras")
wc_model.save(MODELS_DIR / "wasting_classifier.keras")

with open(MODELS_DIR / "feature_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
with open(MODELS_DIR / "label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("Keras models saved.")

# Export TFLite
def export_tflite(model, out_path):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    out_path.write_bytes(converter.convert())
    print(f"  TFLite → {out_path.name}  ({out_path.stat().st_size // 1024} KB)")

export_tflite(we_model, MODELS_DIR / "weight_estimator.tflite")
export_tflite(wc_model, MODELS_DIR / "wasting_classifier.tflite")

print("\nDone. The server will pick up the new models on next restart.")
print("Run: PYTHONPATH=. .venv/bin/python ml/evaluate.py  — to confirm SAM recall")

## Summary

| | Before fine-tuning | After fine-tuning |
|--|--|--|
| Weight estimator val MAE | 0.544 kg | ← see above |
| Classifier val accuracy | 0.578 | ← see above |
| SAM recall | 0.806 | ← see above |

**Tips for improvement:**
- Increase `REAL_DATA_REPEAT` (e.g. 50×) if your real dataset is < 20 samples
- Collect more SAM/MAM cases — they are the rarest and most important
- Ensure photos are frontal, full-body, and well-lit (matches training assumptions)